### initialization

In [ ]:
# basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# models
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer


# dataset
from ucimlrepo import fetch_ucirepo
from statsmodels.stats.outliers_influence import variance_inflation_factor

# other 
import warnings


# Disable warning print
warnings.filterwarnings('ignore')

# global parameters
SEED = 2002

# fetch dataset 
heart_failure_clinical_records = fetch_ucirepo(id=519) 
# data (as pandas dataframes) 
X_GLOBAL = heart_failure_clinical_records.data.features 
Y_GLOBAL = heart_failure_clinical_records.data.targets 

## Dataset 
We analyze a dataset of 299 heart failure patients collected from two hospitals in Faisalabad, Pakistan, between April and December 2015. The dataset includes 13 features covering demographic, clinical, and lifestyle information, such as age, blood pressure, ejection fraction, and smoking status. All patients had left ventricular systolic dysfunction and a history of heart failure classified as NYHA class III or IV. The target variable indicates whether a patient died during a follow-up period, with 32% of patients experiencing a death event.

In [ ]:
print(heart_failure_clinical_records.variables) # information about data set

## Data stucture
The dataset is clean, containing no missing values or duplicates. The dataset includes both numerical and categorical features, with categorical variables already one-hot encoded. We will focus on predicting death events in patients following a heart attack. For modeling, we will apply scaling as needed, particularly for Support Vector Classification (SVC) and Logistic Regression models.

In [ ]:
# Look for missing values/duplicates
print(any(X_GLOBAL.duplicated()))
print(X_GLOBAL.isnull().any())
X_GLOBAL

### SVC part

#### Methodology
- We compared two *SVC* models with different scaling methods to evaluate potential differences in performance.  
- We considered both *radial* and *polynomial* kernels. The *linear* kernel was discarded due to the need for a high *C* value, which could lead to overfitting.  

#### Important properties of X
- The model is sensitive to feature scaling.  
- We used **RobustScaler** based on [1] and [2], which indicate it is a suitable choice for data with outliers. 

---

1. https://users.ece.utexas.edu/~cmcaram/pubs/XuCaramanisMannor.SVM1.pdf?utm_source=chatgpt.com
2. https://scikit-learn.org/1.5/auto_examples/preprocessing/plot_all_scaling.html?utm_source=chatgpt.com

In [ ]:
# creating SVC_part variables 
X = X_GLOBAL.copy()
y = Y_GLOBAL.copy()

# model preparation
svm_models = {
    'SVC model with MinMaxScaler': make_pipeline(MinMaxScaler(), SVC()), 
    'SVC model with RobustScaler': make_pipeline(RobustScaler(),SVC())
}

for key, svm_model in svm_models.items():
    # splitting data 
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=SEED, stratify=y)

    # train model, find right hyperparameters 
    param_grid = {
        'svc__kernel': ['rbf', 'poly'], # we can also add polinomials
        'svc__C': np.logspace(-2, 2, 5), # [0.01, -> 100]
        'svc__gamma': ['scale', 'auto'],
        'svc__degree': [2, 3, 4] 
    }
    grid = GridSearchCV(svm_model, param_grid, cv=5, scoring='accuracy')
    grid.fit(X_train, y_train)
    model = grid.best_estimator_

    # checking metric 
    print(f'{key}\n')
    print(classification_report(y_test, model.predict(X_test)))
    print(f'Model final parameters: \n{grid.best_params_}')
    print('\n---------------------------\n')

    # comments about how we can do better 


## Logistic Regression implementation

#### Data Preprocessing and Class Imbalance

The dataset is slightly imbalanced, with a higher number of patients who survived. To address this, we may apply class weighting to penalize the overrepresented class and help the model better detect cases of death. Since our primary goal is to **minimize false negatives**—i.e., avoid missing patients who may die—this step is particularly important.

All categorical variables, such as `sex` and `smoking`, are already encoded in binary format, so there is no need for additional encoding. We will scale the numerical features using `StandardScaler()` to ensure consistent feature ranges for the model.


In [ ]:
# creating LR_part variables 
X = X_GLOBAL.copy()
y = Y_GLOBAL.copy()

# Exploring dataset
print(Y_GLOBAL.value_counts())
print('Our target dataset is slightly imbalanced with more patients that survived therefore we could consider using penalization for overrepresented class.')
print(X_GLOBAL.info())

In [ ]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=SEED, stratify=y)

# Preprocessing
numerical_columns = ['age', 'creatinine_phosphokinase', 'ejection_fraction', 'platelets', 'serum_creatinine', 'serum_sodium', 'time']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns)
    ],
    remainder='passthrough'  # Keep other columns as-is
)


### VIF Factor
We calculate the Variance Inflation Factor (VIF) to assess multicollinearity among the predictor variables. VIF scores are low.

In [ ]:
X_num = X_train[numerical_columns]
X_scaled = StandardScaler().fit_transform(X_num)
vif_data = pd.DataFrame()
vif_data["feature"] = X_num.columns
vif_data["VIF"] = [variance_inflation_factor(X_scaled, i) for i in range(X_scaled.shape[1])]
print(vif_data)

### Logistic Regression

We explore various regularization techniques for logistic regression, including **Lasso (L1)**, **Ridge (L2)**, and **Elastic Net** (a combination of L1 and L2). The `LogisticRegressionCV()` function in scikit-learn includes built-in cross-validation for hyperparameter tuning, so there is no need to use `GridSearchCV()` separately.


In [ ]:
metric = 'accuracy'
models = {'Ridge regularisation': make_pipeline(preprocessor, LogisticRegressionCV(Cs=10, cv=5, scoring=metric, penalty='l2', solver='lbfgs', random_state=SEED, class_weight='balanced')),
          'Lasso regularisation': make_pipeline(preprocessor, LogisticRegressionCV(Cs=10, cv=5, scoring=metric, penalty='l1', solver='saga', random_state=SEED, class_weight='balanced')),
          'Elastic Net regularisation': make_pipeline(preprocessor, LogisticRegressionCV(Cs=10, cv=5, scoring=metric, penalty='elasticnet', l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9], solver='saga', random_state=SEED, class_weight='balanced'))         
}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    print(name)
    print(classification_report(y_test, predictions))
    print(roc_auc_score(y_test, predictions))
    print(confusion_matrix(y_test, predictions))
    print('\n---------------------------\n')
    


## Conclusion

### SVC – comparing scalers  
The accuracy scores for the SVC models using **MinMaxScaler** and **RobustScaler** are 0.83 and 0.84, respectively. The recall for the positive class (class 1) is 0.62 and 0.71. Both models use the *radial* kernel, with *C* values of 10 and 1, respectively.  

Based on these results, we conclude that the SVC model with **RobustScaler** not only yields better recall but also requires a lower *C* value, which helps reduce the risk of overfitting.  

### Logistic Regression – classification performance  
The logistic regression models using **Ridge** and **Lasso** regularization achieved strong and comparable performance, with accuracies of **0.84** and **0.83**, and AUC scores around **0.82**. The **Elastic Net** model showed slightly lower performance, with an accuracy of **0.80** and an AUC of approximately **0.79**.  

In terms of recall for the positive class (class 1), the **Lasso-regularized model** achieved the highest value of **0.79**, while both **Ridge** and **Elastic Net** reached **0.75**.  


### Final Conclusion  
Based on the results above, we select the **Logistic Regression model with Lasso regularization** as the most suitable choice. This model achieved the **highest recall** for the positive class (class 1), which is critical in our context of predicting **death events**. That is because missing true positive cases may result in the failure to identify individuals at risk, which in turn limits the possibility of **early intervention**.
 